<a href="https://colab.research.google.com/github/SAINIDHI2005/Bluetooth-RC-Car---STM32/blob/main/ids_pcap_v1.2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
!pip install torch-geometric -q

In [ ]:
import pandas as pd
import numpy as np
import torch

from sklearn.preprocessing import StandardScaler

from torch_geometric.data import Data

In [ ]:
import os

DATASET_DIR = "/content/drive/MyDrive/CIC_IoT_2023/dummy"

files = {
    "BENIGN.csv":0,
    "DDoS.csv":1,
    "DoS.csv":1,
    "Mirai.csv":1,
    "Recon.csv":1,
    "Spoofing.csv":1
}

dfs = []

for file,label in files.items():

    path = os.path.join(DATASET_DIR,file)

    df = pd.read_csv(path)

    df["label"] = label

    print(file,df.shape)

    dfs.append(df)

data = pd.concat(
    dfs,
    ignore_index=True
)

print(data.shape)

In [ ]:
data = data.reset_index(drop=True)

data["node_id"] = np.arange(len(data))

print("Nodes:",len(data))

In [ ]:
DROP_COLS = [

    "src_ip",
    "dst_ip",

    "src_mac",
    "dst_mac",

    "src_oui",
    "dst_oui",

    "application_name",
    "application_category_name",

    "requested_server_name",

    "client_fingerprint",
    "server_fingerprint",

    "user_agent",
    "content_type",

    "node_id",
    "label"
]

feature_cols = [
    c for c in data.columns
    if c not in DROP_COLS
]

print("Features:",len(feature_cols))

In [ ]:
for col in feature_cols:

    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )

data[feature_cols] = (
    data[feature_cols]
    .fillna(0)
)

In [ ]:
scaler = StandardScaler()

X = scaler.fit_transform(
    data[feature_cols]
)

print(X.shape)

In [ ]:
src_groups = data.groupby(
    ["src_ip",
     "protocol"]
)["node_id"].apply(list)

print("Source Groups:", len(src_groups))

In [ ]:
dst_groups = data.groupby(
    ["dst_ip",
     "protocol"]
)["node_id"].apply(list)

print("Destination Groups:", len(dst_groups))

In [ ]:
edges = []

In [ ]:
for nodes in src_groups:

    if len(nodes) < 2:
        continue

    for i in range(len(nodes)-1):

        u = nodes[i]
        v = nodes[i+1]

        edges.append([u,v])
        edges.append([v,u])

In [ ]:
K = 7

for nodes in dst_groups:

    if len(nodes) < 2:
        continue

    for i in range(len(nodes)):

        for j in range(
            i + 1,
            min(i + K + 1, len(nodes))
        ):

            u = nodes[i]
            v = nodes[j]

            edges.append([u, v])
            edges.append([v, u])

In [ ]:
edge_index = torch.tensor(
    edges,
    dtype=torch.long
).t().contiguous()

print(edge_index.shape)

In [ ]:
x = torch.tensor(
    X,
    dtype=torch.float
)

print(x.shape)

In [ ]:
y = torch.tensor(
    data["label"].values,
    dtype=torch.long
)

print(y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(data))

train_idx, test_idx = train_test_split(
    idx,
    test_size=0.2,
    random_state=42,
    stratify=y.numpy()
)

In [ ]:
train_mask = torch.zeros(
    len(data),
    dtype=torch.bool
)

test_mask = torch.zeros(
    len(data),
    dtype=torch.bool
)

train_mask[train_idx] = True
test_mask[test_idx] = True

In [ ]:
graph = Data(

    x=x,

    edge_index=edge_index,

    y=y,

    train_mask=train_mask,

    test_mask=test_mask
)

print(graph)

In [ ]:
print(graph)
print(graph.num_nodes)
print(graph.num_edges)

In [ ]:
print(data["label"].value_counts())

In [ ]:
from torch_geometric.utils import degree

deg = degree(
    graph.edge_index[0],
    graph.num_nodes
)

print("Min degree:", deg.min().item())
print("Max degree:", deg.max().item())
print("Mean degree:", deg.float().mean().item())

print(
    "Isolated nodes:",
    (deg==0).sum().item()
)

In [ ]:
print("Train")

print(
    pd.Series(
        graph.y[
            graph.train_mask
        ].numpy()
    ).value_counts()
)

print("\nTest")

print(
    pd.Series(
        graph.y[
            graph.test_mask
        ].numpy()
    ).value_counts()
)

In [ ]:
import torch
import torch.nn.functional as F

from torch_geometric.nn import SAGEConv

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class GraphSAGE(torch.nn.Module):

    def __init__(self,
                 in_channels,
                 hidden_channels,
                 num_classes):

        super().__init__()

        # GraphSAGE Layers
        self.conv1 = SAGEConv(
            in_channels,
            hidden_channels
        )

        self.conv2 = SAGEConv(
            hidden_channels,
            hidden_channels
        )

        self.conv3 = SAGEConv(
            hidden_channels,
            hidden_channels // 2
        )

        # Batch Normalization
        self.bn1 = torch.nn.BatchNorm1d(
            hidden_channels
        )

        self.bn2 = torch.nn.BatchNorm1d(
            hidden_channels
        )

        self.bn3 = torch.nn.BatchNorm1d(
            hidden_channels // 2
        )

        # Classifier
        self.classifier = torch.nn.Linear(
            hidden_channels // 2,
            num_classes
        )

    def forward(self,
                x,
                edge_index):

        x = self.conv1(
            x,
            edge_index
        )

        x = self.bn1(x)

        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.4,
            training=self.training
        )

        x = self.conv2(
            x,
            edge_index
        )

        x = self.bn2(x)

        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.4,
            training=self.training
        )

        x = self.conv3(
            x,
            edge_index
        )

        x = self.bn3(x)

        x = F.relu(x)

        x = self.classifier(x)

        return x

In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

In [ ]:
model = GraphSAGE(
    in_channels=graph.num_node_features,
    hidden_channels=256,
    num_classes=2
)

model = model.to(device)

graph = graph.to(device)

print(model)

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.002,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=20,
    gamma=0.5
)

In [ ]:
criterion = torch.nn.CrossEntropyLoss(
    weight=torch.tensor([1.0, 1.5]).to(device)
)

In [ ]:
def train():

    model.train()

    optimizer.zero_grad()

    out = model(
        graph.x,
        graph.edge_index
    )

    loss = criterion(
        out[graph.train_mask],
        graph.y[graph.train_mask]
    )

    loss.backward()

    optimizer.step()

    return loss.item()

In [ ]:
@torch.no_grad()

def evaluate(mask):

    model.eval()

    out = model(
        graph.x,
        graph.edge_index
    )

    pred = out.argmax(dim=1)

    correct = (
        pred[mask]
        ==
        graph.y[mask]
    ).sum()

    acc = (
        correct.item()
        /
        mask.sum().item()
    )

    return acc

In [ ]:
for epoch in range(1, 11):

    loss = train()

    train_acc = evaluate(
        graph.train_mask
    )

    test_acc = evaluate(
        graph.test_mask
    )

    scheduler.step()

    print(
        f"Epoch {epoch:02d}"
        f" | Loss {loss:.4f}"
        f" | Train {train_acc:.4f}"
        f" | Test {test_acc:.4f}"
    )

In [ ]:
model.eval()

with torch.no_grad():

    out = model(
        graph.x,
        graph.edge_index
    )

    pred = out.argmax(dim=1)

In [ ]:
from sklearn.metrics import (
    classification_report
)

print(

    classification_report(

        graph.y[
            graph.test_mask
        ].cpu(),

        pred[
            graph.test_mask
        ].cpu()
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# True labels
y_true = graph.y[graph.test_mask].cpu().numpy()

# Predicted labels
y_pred = pred[graph.test_mask].cpu().numpy()

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(cm)

# Plot
plt.figure(figsize=(7,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Benign", "Attack"],
    yticklabels=["Benign", "Attack"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("GraphSAGE Confusion Matrix")

plt.show()